In [1]:
import numpy as np
import tensorly as tl
from tensorly.decomposition import parafac

tl.set_backend('numpy')


In [ ]:
def f(xyz, c=5):
    return 1 / (1 + c**2 * np.sum(xyz**2, axis=1))

def chebyshev_polys(x, deg):
    T = np.zeros((deg+1, len(x)))
    T[0] = 1
    if deg > 0:
        T[1] = x
    for k in range(2, deg+1):
        T[k] = 2 * x * T[k-1] - T[k-2]
    return T


In [ ]:
def generate_coeff_tensor(N, c):
    k = np.arange(N+1)
    nodes = np.cos((2*k + 1) * np.pi / (2*(N+1)))
    X, Y, Z = np.meshgrid(nodes, nodes, nodes, indexing="ij")
    coords = np.stack([X.ravel(), Y.ravel(), Z.ravel()], axis=1)
    F = f(coords, c).reshape((N+1, N+1, N+1))

    Tx = chebyshev_polys(nodes, N)
    Ty = Tx.copy()
    Tz = Tx.copy()

    A = np.kron(Tz.T, np.kron(Ty.T, Tx.T))
    F_flat = F.ravel()
    c_flat, *_ = np.linalg.lstsq(A, F_flat, rcond=None)
    C = c_flat.reshape((N+1, N+1, N+1))
    return C, nodes


In [ ]:
def evaluate_cp_interp(weights, factors, nodes, N, resolution=50):
    xx = np.linspace(-1, 1, resolution)
    yy = np.linspace(-1, 1, resolution)
    zz = np.linspace(-1, 1, resolution)
    Tx = chebyshev_polys(xx, N)
    Ty = chebyshev_polys(yy, N)
    Tz = chebyshev_polys(zz, N)
    F_cp = np.zeros((resolution, resolution, resolution))

    for r in range(len(weights)):
        a = Tx.T @ factors[0][:, r]
        b = Ty.T @ factors[1][:, r]
        c = Tz.T @ factors[2][:, r]
        F_cp += weights[r] * np.einsum("i,j,k->ijk", a, b, c)

    return F_cp

def compute_exact_function_grid(f, c, resolution=50):
    xx = np.linspace(-1, 1, resolution)
    yy = np.linspace(-1, 1, resolution)
    zz = np.linspace(-1, 1, resolution)
    X, Y, Z = np.meshgrid(xx, yy, zz, indexing="ij")
    coords = np.stack([X.ravel(), Y.ravel(), Z.ravel()], axis=1)
    F = f(coords, c).reshape((resolution, resolution, resolution))
    return F

def evaluate_direct_chebyshev_interp(C, nodes, N, resolution=50):
    xx = np.linspace(-1, 1, resolution)
    yy = np.linspace(-1, 1, resolution)
    zz = np.linspace(-1, 1, resolution)
    Tx = chebyshev_polys(xx, N)
    Ty = chebyshev_polys(yy, N)
    Tz = chebyshev_polys(zz, N)
    F_interp = np.einsum("ia,jb,kc,abc->ijk", Tx.T, Ty.T, Tz.T, C)
    return F_interp


In [ ]:
N = 63
c = 5
resolution = 100
ranks = range(1, 11)

C, nodes = generate_coeff_tensor(N, c)
F_true = compute_exact_function_grid(f, c, resolution)


In [ ]:
for rank in ranks:
    cp_tensor = parafac(C, rank=rank, init='svd')
    weights, factors = cp_tensor.weights, cp_tensor.factors

    F_cp = evaluate_cp_interp(weights, factors, nodes, N, resolution)

    diff_func = F_true - F_cp
    rmse_func = np.sqrt(np.mean(diff_func**2))
    maxe_func = np.max(np.abs(diff_func))

    C_reconstructed = tl.cp_to_tensor(cp_tensor)
    l2_norm_coeff = np.linalg.norm(C - C_reconstructed)

    print(f"Rank {rank}: RMSE = {rmse_func:.2e}, MaxE = {maxe_func:.2e}, L2 Norm = {l2_norm_coeff:.2e}")


In [ ]:
F_interp_baseline = evaluate_direct_chebyshev_interp(C, nodes, N, resolution)
diff_baseline = F_true - F_interp_baseline
rmse_baseline = np.sqrt(np.mean(diff_baseline**2))
maxe_baseline = np.max(np.abs(diff_baseline))

print(f"[Baseline] RMSE: {rmse_baseline:.2e}, MaxE: {maxe_baseline:.2e}")
